# ResNet101 - KHOTAA Diabetic Foot Ulcer Classification

## 1. Imports & Configuration

In [5]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import resnet101, ResNet101_Weights
import numpy as np
from sklearn.model_selection import StratifiedKFold
import importlib

sys.path.append('../')
sys.path.append('./')

from dataset_loader import SplitFolderDatasetLoader
from dataset_preprocessing import DFUPreprocessing
from utils import checkpoint_manager, training_engine
importlib.reload(checkpoint_manager)
importlib.reload(training_engine)
from utils.checkpoint_manager import CheckpointManager
from utils.training_engine import TrainingEngine, create_optimizer
from utils.metrics_evaluator import (
    calculate_metrics, print_metrics, plot_confusion_matrix,
    plot_roc_curve, plot_training_history
)

print("✓ Imports complete (modules reloaded)")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

✓ Imports complete (modules reloaded)
PyTorch version: 2.8.0
CUDA available: False


## 2. Load Dataset

In [6]:
# Load dataset
loader = SplitFolderDatasetLoader(root_dir='../../dfu-dataset-annotated-into-4-classes')
classes = loader.get_classes()
num_classes = loader.get_num_classes()

print(f"Classes: {classes}")
print(f"Number of classes: {num_classes}")

# Initialize preprocessing
preprocessor = DFUPreprocessing()
train_transform = preprocessor.get_train_transforms()
val_test_transform = preprocessor.get_valid_test_transforms()

# Dataset class
class DFUDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        from PIL import Image
        image = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# Prepare data for cross-validation
X_train, y_train = loader.load_split_paths('train', shuffle=True)
X_val, y_val = loader.load_split_paths('valid')
X_all = np.concatenate([X_train, X_val])
y_all = np.concatenate([y_train, y_val])

# Test set (untouched until final evaluation)
X_test, y_test = loader.load_split_paths('test')
test_dataset = DFUDataset(X_test, y_test, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

# Initialize 5-fold stratified cross-validation
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\nTotal training samples (train+valid): {len(X_all)}")
print(f"Test samples: {len(X_test)}")
print("✓ Dataset loaded and ready for 5-fold cross-validation")

[DatasetLoader] Root: /Users/manaralharbi/Desktop/KHOTAA/dfu-dataset-annotated-into-4-classes
[DatasetLoader] Splits: ['train', 'valid', 'test']
[DatasetLoader] Classes (4): ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4']
Classes: ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4']
Number of classes: 4
[DFUPreprocessing] Initialized
[DFUPreprocessing] Image size: 224x224
[DFUPreprocessing] Train: with augmentation
[DFUPreprocessing] Valid/Test: no augmentation
[DatasetLoader] Split 'train': 9639 images
[DatasetLoader] Split 'valid': 282 images
[DatasetLoader] Split 'test': 141 images

Total training samples (train+valid): 9921
Test samples: 141
✓ Dataset loaded and ready for 5-fold cross-validation


## 3. Model Definition

In [7]:
# Setup device and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()

print(f"Device: {device}")

# Create ResNet101 model
def create_resnet101_model(num_classes=4, pretrained=True):
    """
    Create ResNet101 model for DFU classification.
    
    Args:
        num_classes: Number of output classes (4 for DFU grades)
        pretrained: Use ImageNet pretrained weights
    
    Returns:
        ResNet101 model configured for DFU classification
    """
    if pretrained:
        model = models.resnet101(weights=ResNet101_Weights.IMAGENET1K_V2)
    else:
        model = models.resnet101(weights=None)
    
    # Modify final fully connected layer
    # ResNet101 fc layer: Linear(2048 -> 1000)
    # Replace with: Linear(2048 -> num_classes)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    return model

# Test model creation
test_model = create_resnet101_model(num_classes=num_classes)
print(f"\n✓ ResNet101 model created")
print(f"Input size: 224x224")
print(f"Output classes: {num_classes}")
print(f"Final FC layer: {test_model.fc}")

Device: cpu

✓ ResNet101 model created
Input size: 224x224
Output classes: 4
Final FC layer: Linear(in_features=2048, out_features=4, bias=True)


## 4. Training

In [ ]:
# 5-Fold Cross-Validation Training
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_all, y_all), 1):
    print(f"\n{'='*60}\nFOLD {fold}/5\n{'='*60}")
    
    # Prepare fold data
    X_train_fold = X_all[train_idx]
    y_train_fold = y_all[train_idx]
    X_val_fold = X_all[val_idx]
    y_val_fold = y_all[val_idx]
    
    train_dataset = DFUDataset(X_train_fold, y_train_fold, transform=train_transform)
    val_dataset = DFUDataset(X_val_fold, y_val_fold, transform=val_test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
    
    # Create model
    model = create_resnet101_model(num_classes=num_classes, pretrained=True)
    model = model.to(device)
    
    # Setup optimizer and scheduler
    optimizer = create_optimizer(model, lr=0.001)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
    checkpoint_manager = CheckpointManager(base_dir='checkpoints', experiment_name=f'resnet101_fold{fold}')
    engine = TrainingEngine(model=model, device=device)
    
    # Train
    history = engine.train(
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        num_epochs=30,
        scheduler=scheduler,
        checkpoint_manager=checkpoint_manager,
        early_stopping_patience=7,
        use_early_stopping=True,
        verbose=True
    )
    
    # Store results
    best_val_acc = max(history['val_acc'])
    fold_results.append({
        'fold': fold,
        'best_val_acc': best_val_acc,
        'final_val_acc': history['val_acc'][-1],
        'stopped_epoch': history['stopped_epoch'],
        'history': history
    })
    print(f"Fold {fold} - Best Acc: {best_val_acc*100:.2f}% (stopped at epoch {history['stopped_epoch']})")

# Cross-validation summary
avg_acc = np.mean([r['best_val_acc'] for r in fold_results])
std_acc = np.std([r['best_val_acc'] for r in fold_results])
avg_epochs = np.mean([r['stopped_epoch'] for r in fold_results])

print(f"\n{'='*60}")
print(f"5-FOLD CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(f"Mean Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
print(f"Average Epochs: {avg_epochs:.1f}")
print(f"\nIndividual Fold Results:")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_val_acc']*100:.2f}% (epoch {r['stopped_epoch']})")
print(f"{'='*60}")


FOLD 1/5
Created SGD optimizer: lr=0.001, momentum=0.8, weight_decay=0.0001
Training on: cpu

Epoch 1/30


Evaluating: 100%|██████████| 63/63 [06:55<00:00,  6.60s/it, loss=1.1505, acc=53.30%]


Learning Rate: 0.001000

Epoch 1 Results:
   Train Loss: 1.2725 | Train Acc: 43.36%
   Val Loss:   1.1742 | Val Acc:   53.30%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold1/fold_1/checkpoint_epoch_1.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold1/fold_1/best_accuracy.pt
New best model! Val Acc: 53.30%

Epoch 2/30


Evaluating: 100%|██████████| 63/63 [06:45<00:00,  6.44s/it, loss=0.7342, acc=61.81%]


Learning Rate: 0.001000

Epoch 2 Results:
   Train Loss: 1.0646 | Train Acc: 56.24%
   Val Loss:   0.9748 | Val Acc:   61.81%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold1/fold_1/checkpoint_epoch_2.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold1/fold_1/best_accuracy.pt
New best model! Val Acc: 61.81%

Epoch 3/30


...

Evaluating: 100%|██████████| 62/62 [07:17<00:00,  7.05s/it, loss=0.8376, acc=89.21%]


Learning Rate: 0.000100

Epoch 17 Results:
   Train Loss: 0.2858 | Train Acc: 90.20%
   Val Loss:   0.3593 | Val Acc:   89.21%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold2/fold_1/checkpoint_epoch_17.pth
SUCCESS: Best model saved: checkpoints/resnet101_fold2/fold_1/best_accuracy.pt
New best model! Val Acc: 89.21%

Epoch 18/30


Evaluating: 100%|██████████| 62/62 [07:50<00:00,  7.58s/it, loss=0.7314, acc=88.71%]


Learning Rate: 0.000100

Epoch 18 Results:
   Train Loss: 0.2896 | Train Acc: 89.63%
   Val Loss:   0.3696 | Val Acc:   88.71%
SUCCESS: Checkpoint saved: checkpoints/resnet101_fold2/fold_1/checkpoint_epoch_18.pth

Epoch 19/30


Training:  12%|█▏        | 31/249 [06:44<47:33, 13.09s/it, loss=0.3274, acc=90.12%]

## 5. Evaluation & Plots

In [ ]:
# Test Set Evaluation
print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)

# Check if results already exist
import os
import json

# Organize results by model name
model_results_dir = 'results/resnet101'
results_file = f'{model_results_dir}/resnet101_results.json'

if os.path.exists(results_file):
    print(f"\n✓ Found existing results file: {results_file}")
    print("Loading previously saved results...\n")
    
    with open(results_file, 'r') as f:
        resnet101_results = json.load(f)
    
    # Display the loaded results
    print("="*60)
    print("RESNET101 - LOADED RESULTS")
    print("="*60)
    
    cv_results = resnet101_results['cv_results']
    test_results = resnet101_results['test_results']
    inference = resnet101_results.get('inference_time', {})
    
    print(f"\nCROSS-VALIDATION RESULTS:")
    print(f"   Mean Accuracy: {cv_results['val_accuracy']['mean']*100:.2f}% ± {cv_results['val_accuracy']['std']*100:.2f}%")
    print(f"   Average Epochs: {cv_results['avg_epochs']:.1f}")
    
    print(f"\nTEST SET RESULTS:")
    print(f"   Test Accuracy: {test_results['test_accuracy']*100:.2f}%")
    print(f"   Test Loss: {test_results.get('test_loss', 'N/A'):.4f}" if 'test_loss' in test_results else "   Test Loss: N/A")
    print(f"   Precision: {test_results['precision']:.4f}")
    print(f"   Recall: {test_results['recall']:.4f}")
    print(f"   F1-Score: {test_results['f1_score']:.4f}")
    print(f"   Specificity: {test_results.get('specificity', 'N/A'):.4f}" if 'specificity' in test_results else "   Specificity: N/A")
    print(f"   Sensitivity: {test_results.get('sensitivity', 'N/A'):.4f}" if 'sensitivity' in test_results else "   Sensitivity: N/A")
    print(f"   MCC: {test_results['mcc']:.4f}")
    print(f"   AUC: {test_results['auc']:.4f}")
    
    if inference:
        print(f"\nINFERENCE TIME:")
        print(f"   Avg Time/Image: {inference.get('avg_time_per_image_ms', 'N/A'):.2f}ms")
        print(f"   Throughput: {inference.get('throughput_fps', 'N/A'):.1f} images/second")
    
    print(f"\nINDIVIDUAL FOLD RESULTS:")
    for fold_result in cv_results['fold_results']:
        print(f"   Fold {fold_result['fold']}: {fold_result['best_val_acc']*100:.2f}% (epoch {fold_result['stopped_epoch']})")
    
    print("="*60)
    
    # Check for visualization files
    viz_files = [
        f'{model_results_dir}/confusion_matrix.png',
        f'{model_results_dir}/roc_curve.png',
        f'{model_results_dir}/training_history.png'
    ]
    
    print(f"\nVISUALIZATION FILES:")
    for viz_file in viz_files:
        if os.path.exists(viz_file):
            print(f"   ✓ {viz_file}")
        else:
            print(f"   ✗ {viz_file} (not found)")
    
    print(f"\nTIP: To regenerate results, delete {results_file} and re-run training")
    print("="*60)

else:
    # Original evaluation code - runs only if results don't exist
    print(f"\n  No existing results found at {results_file}")
    print("Running full evaluation...\n")
    
    # Load best fold model
    best_fold_idx = np.argmax([r['best_val_acc'] for r in fold_results])
    best_fold_num = fold_results[best_fold_idx]['fold']

    print(f"Loading best model from Fold {best_fold_num}")

    checkpoint_manager = CheckpointManager(base_dir='checkpoints', experiment_name=f'resnet101_fold{best_fold_num}')

    # Load best model
    model = checkpoint_manager.load_best_model(
        fold_index=0,
        create_model_fn=lambda: create_resnet101_model(num_classes=num_classes, pretrained=False),
        metric_name='accuracy'
    )
    model = model.to(device)

    engine = TrainingEngine(model=model, device=device)

    # Evaluate with inference time tracking
    test_loss, test_acc, predictions, true_labels, inference_time = engine.evaluate(
        test_loader, 
        criterion, 
        measure_inference_time=True
    )

    print(f"\nTest Accuracy: {test_acc*100:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"\nInference Time Statistics:")
    print(f"  Total Time: {inference_time['total_time']:.4f}s")
    print(f"  Avg Time/Batch: {inference_time['avg_time_per_batch']*1000:.2f}ms ± {inference_time['std_time_per_batch']*1000:.2f}ms")
    print(f"  Avg Time/Image: {inference_time['avg_time_per_image']*1000:.2f}ms")
    print(f"  Throughput: {inference_time['images_per_second']:.1f} images/second")

    # Get probabilities for AUC
    model.eval()
    all_probs = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs.to(device))
            probs = torch.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())

    y_pred_proba = np.vstack(all_probs)

    # Calculate all metrics
    metrics = calculate_metrics(
        y_true=true_labels,
        y_pred=predictions,
        y_pred_proba=y_pred_proba,
        class_names=classes,
        average='macro'
    )

    print("\n" + "="*60)
    print_metrics(metrics, title="ResNet101 Test Results")
    print("="*60)

    # Create model-specific results directory
    os.makedirs(model_results_dir, exist_ok=True)

    # Confusion Matrix
    plot_confusion_matrix(
        y_true=true_labels,
        y_pred=predictions,
        class_names=classes,
        normalize=True,
        save_path=f'{model_results_dir}/confusion_matrix.png'
    )
    print(f"\n✓ Confusion matrix saved to {model_results_dir}/confusion_matrix.png")

    # ROC Curve
    plot_roc_curve(
        y_true=true_labels,
        y_pred_proba=y_pred_proba,
        class_names=classes,
        save_path=f'{model_results_dir}/roc_curve.png'
    )
    print(f"✓ ROC curve saved to {model_results_dir}/roc_curve.png")

    # Training History (best fold)
    plot_training_history(
        fold_results[best_fold_idx]['history'],
        save_path=f'{model_results_dir}/training_history.png'
    )
    print(f"✓ Training history saved to {model_results_dir}/training_history.png")

    # Summary for Model Comparison
    print("\n" + "="*60)
    print("SUMMARY FOR MODEL COMPARISON")
    print("="*60)
    print(f"Model: ResNet101")
    print(f"Cross-Validation Accuracy: {avg_acc*100:.2f}% ± {std_acc*100:.2f}%")
    print(f"Test Accuracy: {test_acc*100:.2f}%")
    print(f"Test F1-Score: {metrics['f1_score']:.4f}")
    print(f"Test MCC: {metrics['mcc']:.4f}")
    print(f"Test AUC: {metrics['auc']:.4f}")
    print(f"Average Training Epochs: {avg_epochs:.1f}")
    print(f"Inference Time: {inference_time['avg_time_per_image']*1000:.2f}ms per image")
    print(f"Throughput: {inference_time['images_per_second']:.1f} images/second")
    print("="*60)

## 6. Save Results for Model Comparison

Save the results for later comparison with other models (ResNet50, DenseNet, MobileNet, GoogLeNet, EfficientNetV2S, PFCNN+DRNN).

In [ ]:
# Save results for model comparison
import json
import os

# Organize by model name
model_results_dir = 'results/resnet101'
results_file = f'{model_results_dir}/resnet101_results.json'

if os.path.exists(results_file):
    print(f"✓ Results already saved at {results_file}")
    print(" To regenerate, delete the file and re-run training & evaluation")
else:
    resnet101_results = {
        'model_name': 'ResNet101',
        'cv_results': {
            'val_accuracy': {'mean': float(avg_acc), 'std': float(std_acc)},
            'avg_epochs': float(avg_epochs),
            'fold_results': [
                {
                    'fold': r['fold'],
                    'best_val_acc': float(r['best_val_acc']),
                    'stopped_epoch': int(r['stopped_epoch'])
                }
                for r in fold_results
            ]
        },
        'test_results': {
            'test_accuracy': float(test_acc),
            'test_loss': float(test_loss),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'f1_score': float(metrics['f1_score']),
            'specificity': float(metrics['specificity']),
            'sensitivity': float(metrics['sensitivity']),
            'mcc': float(metrics['mcc']),
            'auc': float(metrics['auc'])
        },
        'inference_time': {
            'total_time': float(inference_time['total_time']),
            'avg_time_per_image_ms': float(inference_time['avg_time_per_image'] * 1000),
            'throughput_fps': float(inference_time['images_per_second'])
        }
    }

    # Save to JSON in model-specific directory
    os.makedirs(model_results_dir, exist_ok=True)
    with open(results_file, 'w') as f:
        json.dump(resnet101_results, f, indent=4)

    print(f"✓ Results saved to {results_file}")
    print("\nThese results can be used with the ModelComparison utility:")
    print("from utils.model_comparison import ModelComparison")
    print("comparison = ModelComparison()")
    print("comparison.add_model_result(**resnet101_results)")
    print("\n✓ ResNet101 training complete!")